# 🧹 Review Data Cleaning & Silver Promotion

> **Layer Transition:** Bronze Final → Silver Cleaned  
> **Source Table:** `yelp_poc_uc.bronze_final.yelp_review` (6,990,280 rows)  
> **Target Table:** `yelp_poc_uc.silver_cleaned.yelp_review`  
> **DQ Reference:** `03_Review_DQ_Issues_and_Solutions.md`

This notebook implements the **Bronze → Silver Cleaned data population** for the Yelp Review dataset. It applies the chosen resolutions from the Data Quality Issues & Solutions document to produce a clean, type-corrected, NLP-ready Silver Cleaned table.

> **⚠️ Prerequisites — Run These First (in order):**  
> 1. **`01_Business_Silver_Cleaned`** (DDL query) — creates `yelp_poc_uc.silver_cleaned` schema and the empty `yelp_business` table structure (FK dependency)  
> 2. **`01_Business_Cleaning`** (notebook) — populates `yelp_poc_uc.silver_cleaned.yelp_business` with the cleaned business records. It quarantines 15 out-of-scope businesses (DQ-BUSINESS-009 / 010) to `quarantine.yelp_business_out_of_scope` — their reviews must be quarantined here too, so re-run this notebook after every business load  
> 3. **`03_Review_Silver_Cleaned`** (DDL query) — creates the empty `yelp_poc_uc.silver_cleaned.yelp_review` table with PK/FK constraints  
>  
> This notebook then **populates** the review table created by step 3. The DDL queries are idempotent and can be re-run safely.

## 📍 Pipeline Context

| Step | Notebook | UC Location | Layer |
| ---: | --- | --- | --- |
| 1 | `03_Extract_Yelp_Dataset_from_Kaggle` | `/Volumes/yelp_poc_uc/bronze_landing/landing_zone/yelp_textual_dataset/` | Landing Zone |
| 2 | `01_Landing_to_Staging_Schema_Validator` | `yelp_poc_uc.bronze_staging` (`yelp_review_clean` / `yelp_review_errors`) | Bronze Staging |
| 3 | `02_Staging_Quality_Gate_and_Promotion` | `yelp_poc_uc.bronze_final.yelp_review` | Bronze Final |
| 4 | `4.1.3_Review_Basic_Profile` | _(reads bronze_final)_ | Basic EDA |
| 5 | `4.2.3_Review_Deep_EDA` | _(reads bronze_final)_ | Deep EDA / DQ Assessment |
| 6a | `01_Business_Silver_Cleaned` (DDL) | `yelp_poc_uc.silver_cleaned.yelp_business` (schema) | Silver Cleaned DDL |
| 6b | `01_Business_Cleaning` | `yelp_poc_uc.silver_cleaned.yelp_business` (data) | Silver Cleaned |
| 6c | `02_Checkin_Silver_Cleaned` (DDL) | `yelp_poc_uc.silver_cleaned.yelp_checkin` (schema) | Silver Cleaned DDL |
| 6d | `02_Checkin_Cleaning` | `yelp_poc_uc.silver_cleaned.yelp_checkin` (data) | Silver Cleaned |
| 6e | `03_Review_Silver_Cleaned` (DDL) | `yelp_poc_uc.silver_cleaned.yelp_review` (schema) | Silver Cleaned DDL |
| **6f** | **`03_Review_Cleaning` ◄ THIS NOTEBOOK** | **`yelp_poc_uc.silver_cleaned.yelp_review` (data)** | **Silver Cleaned** |
| 7 | Gold Aggregation _(planned)_ | `yelp_poc_uc.gold` _(planned)_ | Gold |

## ✅ Scope — Core DQ Issues Resolved in This Notebook

This notebook addresses the **structural, integrity, and text-quality** issues that must be fixed to produce a valid, analytics-ready Silver table.

### Output Schema

| # | Column | Type | Description |
| --- | --- | --- | --- |
| 1 | `review_id` | `string` | Unique review identifier (PK) |
| 2 | `user_id` | `string` | User identifier (FK to yelp_user) |
| 3 | `business_id` | `string` | Business identifier (FK to yelp_business) |
| 4 | `stars` | `int` | Star rating (1–5), cast from double |
| 5 | `useful` | `int` | Useful vote count (clamped ≥ 0) |
| 6 | `funny` | `int` | Funny vote count (clamped ≥ 0) |
| 7 | `cool` | `int` | Cool vote count (clamped ≥ 0) |
| 8 | `text` | `string` | Review text (cleaned: whitespace normalised, control chars stripped, HTML unescaped, masked nulls → NULL) |
| 9 | `review_timestamp` | `timestamp` | Review timestamp, cast from string `date` column |
| 10 | `is_latest_review` | `boolean` | True if this is the most recent review by this user for this business (SCD pattern — all historical versions preserved) |

No `_ingestion_timestamp` column — lineage is tracked via TBLPROPERTIES metadata (see DDL rule R6).

### DQ Issues Addressed

| DQ Issue | Summary | Resolution Applied |
| --- | --- | --- |
| **DQ-REVIEW-001** | `stars` stored as `double` instead of `int` | Cast to `IntegerType` — all 6.99M values are whole numbers |
| **DQ-REVIEW-002** | `date` stored as `string` instead of `timestamp` | Cast to `TimestampType` via `to_timestamp()` → renamed to `review_timestamp` |
| **DQ-REVIEW-003 / 016** | Negative vote counts (`-1`) in `useful`, `funny`, `cool` | Clamp to 0 with `greatest(col, 0)` |
| **DQ-REVIEW-007** | Multiple consecutive spaces in review text (34% of rows) | Regex collapse: `'  +'` → `' '` |
| **DQ-REVIEW-008** | Control characters in review text (18 rows) | Strip non-printable chars via regex |
| **DQ-REVIEW-009** | HTML entities in review text (97 rows) | Decode common HTML entities (`&amp;` → `&`, etc.) |
| **DQ-REVIEW-010** | Excessive newlines & carriage returns | Collapse `\n{3,}` → `\n\n`, normalise `\r\n` / `\r` → `\n` |
| **DQ-REVIEW-011** | Masked null tokens (`*`, `Tbd`, `.`) in text (11 rows) | Convert to proper NULL |
| **DQ-REVIEW-005** | Very short reviews < 10 chars (156 rows) | Nullify 34 noise records (punct-only/single letters); flag remaining \~112 micro-reviews with `is_short_review` |
| **DQ-REVIEW-014** | Multi-review user-business pairs (244K excess reviews, 212K pairs) | Add `is_latest_review` boolean flag (SCD pattern); all historical versions preserved — `ROW_NUMBER()` over `review_timestamp DESC, review_id DESC`. RCA: 89.3% have 2 versions, median gap 160 days, 41.7% changed rating |
| **DQ-REVIEW-017** | User-ID / Business-ID namespace collision (1 self-referential row) | Quarantine to `quarantine.yelp_review_id_collision`; no flag column — RCA: ID `lJzfEleeQ1uYnYQp37LD9Q` exists in both `yelp_user` and `yelp_business` (genuine hash collision). Broader 3,204 cross-namespace collisions documented only |
| **DQ-REVIEW-019** | Orphan user IDs — 32 users missing from `yelp_user` (33 rows, all on 2022-01-19) | Quarantine to `quarantine.yelp_review_orphan_user`; source-level Yelp export timing gap — 100% zero engagement, all business_ids valid, not a pipeline issue |

## 🔜 Out of Scope — Deferred to Analytics Phase

The DQ document identifies several **advisory and informational findings** that do not require Silver-layer transformation. These are deferred to the **analytics or Gold-layer phase**, where the specific use case will determine if action is needed.

| Deferred DQ Issue | What It Recommends | Why Deferred |
| --- | --- | --- |
| DQ-REVIEW-004 / 005 | Extreme vote outliers & winsorization / log-transform | Feature engineering — applied per-model at Gold layer |
| DQ-REVIEW-006 | Placeholder/test reviews (18 rows) — flag or exclude from NLP | Analytical filter — context-dependent |
| DQ-REVIEW-012 | `is_complete_year` flag (2005/2022 partial years) | Temporal filter — applied at query time |
| DQ-REVIEW-013 | Sparse early years (2005–2008) — minimum-volume thresholds | Analytical context — no schema change needed |
| DQ-REVIEW-015 | Near-duplicate timestamp collisions | Subsumed by DQ-REVIEW-014 `is_latest_review` flag |
| DQ-REVIEW-018 | Disproportionate cool votes (915 rows) | Legitimate voting behavior — document only |
| DQ-REVIEW-020 | Low review↔tip user overlap (15.2%) | Behavioral gap — LEFT JOIN strategy at query time |
| DQ-REVIEW-021 | Cross-table aggregate drift (business & user counters) | Expected for geographic subset — document only |
| DQ-REVIEW-022–025 | Temporal distribution patterns & lifecycle segmentation | Analytical enrichment — belongs in Gold layer |
| DQ-REVIEW-026 | Historical snapshot staleness (data ends 2022-01-19) | Metadata caveat — no row-level transformation |
| DQ-REVIEW-027 | Missing row-level ingestion metadata | Observability gap — tracked via TBLPROPERTIES |

In [0]:
# ============================================================
# 🔧 Environment Setup — Prerequisite Check
# ============================================================
# PREREQUISITES (run these BEFORE this notebook, in order):
#   1. 01_Business_Silver_Cleaned (DDL) — creates silver_cleaned
#      schema and empty yelp_business table structure
#   2. 01_Business_Cleaning (notebook) — populates yelp_business
#      with business IDs (required for FK referential integrity)
#   3. 03_Review_Silver_Cleaned (DDL) — creates the empty
#      silver_cleaned.yelp_review table with PK/FK constraints
#
# This notebook POPULATES the review table created by step 3.
# No schema creation is needed here.

# Verify the target table exists (created by DDL query)
try:
    tbl_info = spark.sql("DESCRIBE TABLE yelp_poc_uc.silver_cleaned.yelp_review").collect()
    print("✅ Target table exists: yelp_poc_uc.silver_cleaned.yelp_review")
    print(f"   Columns: {len([r for r in tbl_info if r['col_name'] and not r['col_name'].startswith('#')])}")
except Exception as e:
    raise RuntimeError(
        "❌ Target table yelp_poc_uc.silver_cleaned.yelp_review does not exist!\n"
        "   Run the prerequisites first (in order):\n"
        "   1. 01_Business_Silver_Cleaned (DDL)\n"
        "   2. 01_Business_Cleaning (notebook)\n"
        "   3. 03_Review_Silver_Cleaned (DDL)\n"
        f"   Error: {e}"
    )

---

## 🔧 Cleaning Pipeline

The pipeline below applies the DQ resolutions in six phases:

1. **Read & Type-Fix** (DQ-001 / DQ-002) — Read `bronze_final.yelp_review`, cast `stars` from `double` → `int`, cast `date` from `string` → `timestamp` (renamed to `review_timestamp`)
2. **Vote Clamping** (DQ-003 / DQ-016) — Clamp `useful`, `funny`, `cool` to ≥ 0 using `greatest(col, 0)` to fix the 7 rows with `-1` values
3. **Text Cleaning** (DQ-007 / DQ-008 / DQ-009 / DQ-010 / DQ-011 / DQ-005) — Multi-pass regex chain on the `text` column: collapse consecutive spaces, strip control characters, decode HTML entities, normalise newlines/CRs, convert masked-null tokens to proper NULL, and nullify noise short-review text (punctuation-only and single-letter strings < 10 chars)
4. **Flag Columns** (DQ-014 / DQ-017 / DQ-019) — Add `is_latest_review` (window function over `user_id, business_id`), quarantine `is_id_collision` (user_id == business_id) and `is_orphan_user` (anti-join against `yelp_user`)
5. **Populate** — `INSERT INTO` the DDL-created `silver_cleaned.yelp_review` table
6. **Validate** — Post-write checks: row count, nulls, types, flag distributions, vote ranges

In [0]:
from pyspark.sql.functions import (
    col, to_timestamp, greatest, lit, count, length,
    regexp_replace, trim, when, lower,
    row_number, current_timestamp,
    sum as _sum, max as _max, min as _min, avg as _avg
)
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType

# ============================================================
# Phase 1: Read Bronze & Type Fixes (DQ-REVIEW-001 / DQ-REVIEW-002)
# ============================================================
# DQ-REVIEW-001: stars is double → cast to int (ordinal 1–5)
# DQ-REVIEW-002: date is string → cast to timestamp, rename to
#                review_timestamp

df_bronze = spark.read.table("yelp_poc_uc.bronze_final.yelp_review")
bronze_count = df_bronze.count()

df_typed = (
    df_bronze
    .withColumn("stars", col("stars").cast(IntegerType()))
    .withColumn("review_timestamp", to_timestamp(col("date"), "yyyy-MM-dd HH:mm:ss"))
    .drop("date")
)

# Validation: check for failed casts
null_stars = df_typed.filter(col("stars").isNull()).count()
null_ts    = df_typed.filter(col("review_timestamp").isNull()).count()

print(f"📥 Bronze source:           {bronze_count:>12,} rows")
print(f"✅ stars cast to int:       {null_stars:>12,} failed casts (expected 0)")
print(f"✅ date → review_timestamp:  {null_ts:>12,} failed casts (expected 0)")
assert null_stars == 0, "❌ stars cast produced NULLs!"
assert null_ts == 0, "❌ date → timestamp cast produced NULLs!"

In [0]:
# ============================================================
# Phase 2: Vote Clamping (DQ-REVIEW-003 / DQ-REVIEW-016)
# ============================================================
# 7 rows have vote counts of -1 in useful/funny/cool columns.
# Yelp vote counts are non-negative integers; -1 is a sentinel
# value (likely system placeholder or vote invalidation).
# Fix: clamp to 0 with greatest(col, 0) — idempotent and safe
# for all 6.99M rows (no-op on values already ≥ 0).

VOTE_COLS = ["useful", "funny", "cool"]

df_clamped = df_typed
for c in VOTE_COLS:
    df_clamped = df_clamped.withColumn(c, greatest(col(c).cast(IntegerType()), lit(0)))

# Report: how many rows were affected per column
for c in VOTE_COLS:
    neg_count = df_typed.filter(col(c) < 0).count()
    print(f"   {c:>6}: {neg_count} rows clamped from -1 → 0")

print("✅ Phase 2 complete — vote columns clamped to ≥ 0")

In [0]:
# ============================================================
# Phase 3: Text Cleaning
# ============================================================
# Multi-pass regex chain on the `text` column:
#   DQ-REVIEW-010: Normalise \r\n → \n, \r → \n, collapse \n{3,} → \n\n
#   DQ-REVIEW-008: Strip non-printable control characters
#   DQ-REVIEW-009: Decode common HTML entities
#   DQ-REVIEW-007: Collapse consecutive spaces (34% of rows)
#   DQ-REVIEW-011: Convert masked-null tokens & noise placeholders to proper NULL
#   DQ-REVIEW-005: Nullify noise short-review text (punct-only / single letters)
#
# Order matters: CR/LF first, then control chars, then HTML
# entities, then spaces (HTML decode can introduce spaces),
# then masked-null / placeholder detection, then noise short-review nullification.

# --- Step 1: Newline / carriage return normalisation (DQ-010) ---
df_text = df_clamped
df_text = df_text.withColumn("text", regexp_replace(col("text"), "\\r\\n", "\n"))
df_text = df_text.withColumn("text", regexp_replace(col("text"), "\\r", "\n"))
df_text = df_text.withColumn("text", regexp_replace(col("text"), "\n{3,}", "\n\n"))

# --- Step 2: Strip control characters (DQ-008) ---
# Remove \x00-\x08, \x0B, \x0C, \x0E-\x1F, \x7F (preserve \n and \t)
df_text = df_text.withColumn(
    "text",
    regexp_replace(col("text"), "[\\x00-\\x08\\x0B\\x0C\\x0E-\\x1F\\x7F]", "")
)

# --- Step 3: Decode common HTML entities (DQ-009) ---
df_text = df_text.withColumn("text", regexp_replace(col("text"), "&amp;", "&"))
df_text = df_text.withColumn("text", regexp_replace(col("text"), "&lt;", "<"))
df_text = df_text.withColumn("text", regexp_replace(col("text"), "&gt;", ">"))
df_text = df_text.withColumn("text", regexp_replace(col("text"), "&nbsp;", " "))
df_text = df_text.withColumn("text", regexp_replace(col("text"), "&quot;", '"'))
df_text = df_text.withColumn("text", regexp_replace(col("text"), "&#39;", "'"))
df_text = df_text.withColumn("text", regexp_replace(col("text"), "&#38;", "&"))

# --- Step 4: Collapse consecutive spaces (DQ-007) ---
df_text = df_text.withColumn("text", regexp_replace(col("text"), "  +", " "))

# --- Step 5: Trim leading/trailing whitespace ---
df_text = df_text.withColumn("text", trim(col("text")))

# --- Step 6: Convert masked-null tokens to NULL (DQ-011) ---
# Tokens identified as masked nulls, placeholders, or zero-semantic-value
# strings (case-insensitive after trim). Includes original 3 plus expanded
# list from manual review of short/noise text.
MASKED_NULLS = [
    "*", "tbd", ".", "..", "...", "......", ".......",
    "hhjjgfhh", "edited", "removed", "lll", "ii",
    "", "update", "see below", "yo", "thanks", "!!",
    "y", "n", "o", "h", "k", "s", "x", "m", "l", "q", "d",
]
df_text = df_text.withColumn(
    "text",
    when(
        lower(trim(col("text"))).isin(MASKED_NULLS) | col("text").isNull(),
        lit(None).cast("string")
    ).otherwise(col("text"))
)

# Count after DQ-011 (before DQ-005)
nulls_after_dq011 = df_text.filter(col("text").isNull()).count()
bronze_null_count = df_bronze.filter(col("text").isNull()).count()
dq011_new_nulls = nulls_after_dq011 - bronze_null_count

# --- Step 7: Nullify noise short-review text (DQ-005) ---
# 156 reviews have text < 10 chars. Of these:
#   - Most true noise is now already NULLified in Step 6 via the
#     expanded MASKED_NULLS list (single letters, placeholders, etc.).
#   - The regex below catches any remaining punct-only or single-
#     letter strings not explicitly listed in MASKED_NULLS.
#   - 2 Chinese-language reviews (沸腾水煮鱼, 饺子还不错) are safe:
#     CJK chars do NOT match the ASCII punct/letter regexes.
#   - ~112 micro-reviews with real sentiment ("Great!", "Closed",
#     "Nope.", "Awesome") are preserved — flagged later in Phase 4
#     with is_short_review for optional NLP filtering.
#
# Noise detection (only applied to text < 10 chars):
#   Pattern A: punctuation/symbol-only  → ^[.!?*:()\s-]+$
#   Pattern B: single ASCII letter      → ^[A-Za-z]$

df_text = df_text.withColumn(
    "text",
    when(
        (length(col("text")) < 10) & (
            col("text").rlike("^[.!?*:()\\s-]+$") |  # punct-only noise
            col("text").rlike("^[A-Za-z]$")           # single ASCII letter
        ),
        lit(None).cast("string")
    ).otherwise(col("text"))
)

# Report
nulls_after_dq005 = df_text.filter(col("text").isNull()).count()
dq005_new_nulls = nulls_after_dq005 - nulls_after_dq011
total_new_nulls = nulls_after_dq005 - bronze_null_count

print("✅ Phase 3 complete — text column cleaned")
print(f"   DQ-011 masked-null tokens → NULL:     {dq011_new_nulls}")
print(f"   DQ-005 noise short reviews → NULL:    {dq005_new_nulls}")
print(f"   Total new NULLs from text cleaning:    {total_new_nulls}")
print(f"   Total NULL text rows (incl. original): {nulls_after_dq005}")

In [0]:
# ============================================================
# Phase 3.5: Quarantine NULL-Text Reviews
# ============================================================
# After text cleaning (Phase 3), rows with NULL text have no
# analytical or NLP value for Silver Cleaned. These include:
#   - Original NULLs from bronze (if any)
#   - Masked-null tokens converted to NULL (DQ-011)
#   - Noise short-review text nullified (DQ-005)
#
# These rows are quarantined to a dedicated table for audit
# and traceability, then removed from the Silver pipeline.

# Split on NULL text
df_quarantine = df_text.filter(col("text").isNull())
df_clean = df_text.filter(col("text").isNotNull())

quarantine_count = df_quarantine.count()
clean_count = df_clean.count()

# Write quarantine rows with a reason column for traceability
(
    df_quarantine
    .withColumn("quarantine_reason", lit("NULL text after cleaning (DQ-005/DQ-011)"))
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("yelp_poc_uc.quarantine.yelp_review_null_text")
)

# Prepare df_flagged for Phase 4: clean rows only
df_flagged = df_clean

# Reconciliation check
assert quarantine_count + clean_count == bronze_count, (
    f"❌ Split mismatch! Quarantine({quarantine_count}) + Clean({clean_count}) != Bronze({bronze_count})"
)

print(f"🔀 Quarantine split complete:")
print(f"   NULL-text → quarantine: {quarantine_count:>12,} rows")
print(f"   Valid text → silver:    {clean_count:>12,} rows")
print(f"   Total:                  {quarantine_count + clean_count:>12,} (= bronze: {bronze_count:,})")
print(f"✅ Quarantine table: yelp_poc_uc.quarantine.yelp_review_null_text")

In [0]:
# ============================================================
# Phase 4: Flags & Quarantines
# ============================================================
# Ref: 03_Review_DQ_Issues_and_Solutions.md (Chosen Approaches)

# DQ-REVIEW-014 (Chosen: Option 2 — Add `is_latest_review` flag):
#   SCD pattern — users revise reviews over time; only the latest
#   version represents the current state. Uses ROW_NUMBER() with
#   review_timestamp DESC; ties broken by review_id DESC.
#   RCA: 89.3% of multi-review pairs have exactly 2 versions
#   (single-edit pattern), median gap 160 days, 41.7% changed
#   their star rating. Dataset design characteristic (retained
#   historical edits), not a defect. All versions preserved.
#
# DQ-REVIEW-017 (Chosen: Quarantine self-referential row):
#   The 1 row where user_id == business_id (lJzfEleeQ1uYnYQp37LD9Q)
#   is quarantined to quarantine.yelp_review_id_collision. No flag
#   column added. RCA: ID exists in both yelp_user (Karen, yelping
#   since 2011) and yelp_business (Dollar Car Rental, Nashville TN).
#   Standard joins work for both; quarantine targets only the
#   self-referential row where dedup/graph logic would break.
#   Broader 3,204 cross-namespace collisions documented as known
#   pattern — no action required.
#
# DQ-REVIEW-019 (Chosen: Quarantine 33 orphan-user rows):
#   32 user_ids (33 rows) with no parent in yelp_user are
#   quarantined to quarantine.yelp_review_orphan_user. RCA: all 33
#   reviews on exact same date (2022-01-19) at dataset boundary.
#   100% zero engagement. All business_ids valid. Source-level Yelp
#   export timing gap, not a pipeline issue.


# --- is_latest_review (DQ-014) ---
w = Window.partitionBy("user_id", "business_id").orderBy(
    col("review_timestamp").desc(), col("review_id").desc()
)
df_flagged = df_flagged.withColumn(
    "_rn", row_number().over(w)
).withColumn(
    "is_latest_review", col("_rn") == 1
).drop("_rn")

# --- Identify orphan users for quarantine (DQ-019) ---
df_users = spark.read.table("yelp_poc_uc.bronze_final.yelp_user").select(
    col("user_id").alias("_uid")
).distinct()

df_flagged = (
    df_flagged
    .join(df_users, df_flagged["user_id"] == df_users["_uid"], "left")
    .withColumn("is_orphan_user", col("_uid").isNull())
    .drop("_uid")
)

# --- Quarantine DQ-017 collision rows ---
df_q_collision = df_flagged.filter(col("user_id") == col("business_id"))
collision_ct = df_q_collision.count()
(
    df_q_collision
    .withColumn("quarantine_reason", lit("user_id == business_id namespace collision (DQ-017)"))
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("yelp_poc_uc.quarantine.yelp_review_id_collision")
)

# --- Quarantine DQ-019 orphan-user rows ---
df_q_orphan = df_flagged.filter(col("is_orphan_user"))
orphan_ct = df_q_orphan.count()
(
    df_q_orphan
    .withColumn("quarantine_reason", lit("user_id not in yelp_user (DQ-019)"))
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("yelp_poc_uc.quarantine.yelp_review_orphan_user")
)

# --- Remove quarantined rows from Silver pipeline ---
df_flagged = df_flagged.filter(
    (col("user_id") != col("business_id")) & ~col("is_orphan_user")
)

# Report
total_flagged = df_flagged.count()
latest_true  = df_flagged.filter(col("is_latest_review")).count()

print("✅ Phase 4 complete — is_latest_review flag added, collision & orphan rows quarantined")
print(f"   is_latest_review = True:  {latest_true:>12,}  (unique user-business pairs)")
print(f"   is_latest_review = False: {total_flagged - latest_true:>12,}  (older revisions)")
print(f"   Quarantined (DQ-017):     {collision_ct:>12,}  (id collision)")
print(f"   Quarantined (DQ-019):     {orphan_ct:>12,}  (orphan users)")
print(f"   Rows continuing to Silver:{total_flagged:>12,}")

---

## 🔗 Phase 4.5 — Business FK Check: Quarantine Reviews of Quarantined / Unknown Businesses (DQ-REVIEW-028)

**The problem:** `01_Business_Cleaning` moves 15 out-of-scope businesses (DQ-BUSINESS-009 / 010: non-existent state `XMS`, address vs coordinates conflict, fully outside the footprint) to `quarantine.yelp_business_out_of_scope`, and they are **not** written to `silver_cleaned.yelp_business`. Their **199 reviews** would point to a `business_id` with no parent row (orphan / FK violation) and would still count stars for a business whose location cannot be trusted.

**Solution taken:** every review whose `business_id` is not in `silver_cleaned.yelp_business` is copied unchanged to `yelp_poc_uc.quarantine.yelp_review_orphan_business` and removed from the Silver pipeline.

| Column | Content |
| --- | --- |
| all Silver review columns | the review as cleaned up to Phase 4 |
| `quarantine_reason` | `PARENT_BUSINESS_QUARANTINED` (business is in the business quarantine table) or `BUSINESS_NOT_IN_SILVER` (unknown business) |
| `reason_detail` | full plain-text explanation: which business, why it was quarantined, the parent's evidence, why the review cannot stay |
| `parent_*` | the parent business's name, reason, DQ issue, pattern and evidence |
| `source_table`, `detected_in`, `action`, `quarantined_at` | lineage of the quarantine decision |

Same table and columns as the one written by the final write step of `01_Business_Cleaning`, so either notebook produces the same quarantine content.

> ⚠️ **Run order:** run this notebook **after** the final write of `01_Business_Cleaning`. The check uses the current `silver_cleaned.yelp_business`; if that table still contains the 15 businesses, it finds 0 orphans.

In [0]:
# ============================================================
# Phase 4.5: Business FK Check — Quarantine Orphan-Business Reviews (DQ-REVIEW-028)
# ============================================================
# Ref: 03_Review_DQ_Issues_and_Solutions.md → DQ-REVIEW-028
#      01_Business_DQ_Issues_and_Solutions.md → DQ-BUSINESS-009 / 010
# Reviews whose business_id is not in silver_cleaned.yelp_business are quarantined to
# quarantine.yelp_review_orphan_business with a detailed reason, then removed from the
# Silver pipeline (no orphan review can reach Silver).
from pyspark.sql import functions as F

SILVER_BUSINESS_TABLE   = "yelp_poc_uc.silver_cleaned.yelp_business"
BUSINESS_QUARANTINE_TBL = "yelp_poc_uc.quarantine.yelp_business_out_of_scope"
ORPHAN_BUSINESS_TBL     = "yelp_poc_uc.quarantine.yelp_review_orphan_business"
REVIEW_COLUMNS = ["review_id", "user_id", "business_id", "stars", "useful", "funny", "cool",
                  "review_timestamp", "text", "is_latest_review"]   # same order as silver_cleaned.yelp_review

df_sv_business = spark.read.table(SILVER_BUSINESS_TABLE).select("business_id").distinct()

# Parent = the quarantined business (reason and evidence), if it is in the business quarantine table
_PARENT_SCHEMA = ("business_id string, parent_business_name string, parent_quarantine_reason string, "
                  "parent_dq_issue string, parent_pattern string, parent_evidence string")
if spark.catalog.tableExists(BUSINESS_QUARANTINE_TBL):
    df_parent = (spark.read.table(BUSINESS_QUARANTINE_TBL)
                 .where(F.col("action") == "MOVED_TO_QUARANTINE")
                 .select("business_id",
                         F.col("name").alias("parent_business_name"),
                         F.col("quarantine_reason").alias("parent_quarantine_reason"),
                         F.col("dq_issue").alias("parent_dq_issue"),
                         F.col("pattern").alias("parent_pattern"),
                         F.col("note").alias("parent_evidence")))
else:
    df_parent = spark.createDataFrame([], _PARENT_SCHEMA)

_v = lambda c: F.coalesce(F.col(c).cast("string"), F.lit("n/a"))
_has_parent = F.col("parent_quarantine_reason").isNotNull()
_reason_parent = F.concat(
    F.lit("PARENT_BUSINESS_QUARANTINED (orphan review). This review (review_id '"), _v("review_id"),
    F.lit("') belongs to business_id '"), _v("business_id"), F.lit("' ('"), _v("parent_business_name"),
    F.lit(f"'), which 01_Business_Cleaning moved to {BUSINESS_QUARANTINE_TBL} with reason "),
    _v("parent_quarantine_reason"), F.lit(" ("), _v("parent_dq_issue"), F.lit(", "), _v("parent_pattern"),
    F.lit("; parent evidence: "), _v("parent_evidence"),
    F.lit("). The business is not in silver_cleaned.yelp_business, so keeping this review in Silver would "
          "leave a business_id without a parent row (orphan / foreign-key violation) and would count stars "
          "and reviews for a business whose location cannot be trusted or lies outside the dataset scope. "
          "The review is kept here unchanged (as cleaned up to Phase 4) so it can be restored together "
          "with its business."))
_reason_unknown = F.concat(
    F.lit("BUSINESS_NOT_IN_SILVER (orphan review). This review (review_id '"), _v("review_id"),
    F.lit("') belongs to business_id '"), _v("business_id"),
    F.lit(f"', which does not exist in {SILVER_BUSINESS_TABLE} and is not in {BUSINESS_QUARANTINE_TBL} "
          "either. Keeping it in Silver would violate the foreign key review.business_id -> "
          "yelp_business.business_id. Check bronze_final.yelp_business and 01_Business_Cleaning for this "
          "business_id. The review is kept here unchanged (as cleaned up to Phase 4)."))

_rows_before_fk = df_flagged.count()

df_q_orphan_biz = (
    df_flagged.join(df_sv_business, "business_id", "left_anti")
    .join(F.broadcast(df_parent), "business_id", "left")
    .withColumns({
        "quarantine_reason": F.when(_has_parent, F.lit("PARENT_BUSINESS_QUARANTINED"))
                              .otherwise(F.lit("BUSINESS_NOT_IN_SILVER")),
        "reason_detail":     F.when(_has_parent, _reason_parent).otherwise(_reason_unknown),
        "dq_issue":          F.coalesce(F.col("parent_dq_issue"), F.lit("DQ-REVIEW-028")),
        "source_table":      F.lit("yelp_poc_uc.bronze_final.yelp_review (cleaned by 03_Review_Cleaning)"),
        "detected_in":       F.lit("03_Review_Cleaning · Phase 4.5 · business FK check"),
        "action":            F.lit("EXCLUDED_FROM_SILVER_YELP_REVIEW"),
        "quarantined_at":    F.current_timestamp(),
    })
    .select(*REVIEW_COLUMNS, "quarantine_reason", "reason_detail", "dq_issue",
            "parent_business_name", "parent_quarantine_reason", "parent_dq_issue",
            "parent_pattern", "parent_evidence", "source_table", "detected_in",
            "action", "quarantined_at")
)

(
    df_q_orphan_biz
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(ORPHAN_BUSINESS_TBL)
)
orphan_biz_ct = spark.read.table(ORPHAN_BUSINESS_TBL).count()

# --- Remove orphan-business reviews from the Silver pipeline ---
df_flagged = df_flagged.join(df_sv_business, "business_id", "left_semi")
_rows_after_fk = df_flagged.count()

assert _rows_before_fk == _rows_after_fk + orphan_biz_ct, (
    f"❌ FK split mismatch: before {_rows_before_fk:,} != kept {_rows_after_fk:,} + quarantined {orphan_biz_ct:,}")

print("✅ Phase 4.5 complete — business FK check")
print(f"   Rows before FK check:         {_rows_before_fk:>12,}")
print(f"   Quarantined (orphan business): {orphan_biz_ct:>11,}  -> {ORPHAN_BUSINESS_TBL}")
print(f"   Rows continuing to Silver:    {_rows_after_fk:>12,}")
display(spark.read.table(ORPHAN_BUSINESS_TBL)
        .groupBy("quarantine_reason", "parent_quarantine_reason", "business_id", "parent_business_name")
        .count().orderBy("quarantine_reason", F.col("count").desc()))


In [0]:
# ============================================================
# Phase 5: Populate Silver Cleaned Table
# ============================================================
# The target table was created by the 03_Review_Silver_Cleaned
# DDL query. We TRUNCATE then append to preserve the DDL-defined
# PK/FK/NOT NULL constraints and TBLPROPERTIES.
# ⚠ Never use mode("overwrite") + overwriteSchema=true here —
#   that replaces the entire table schema, destroying constraints.

# Note: is_orphan_user and is_id_collision removed — those rows
# were quarantined in Phase 4 and are no longer in df_flagged.
SILVER_COLUMNS = [
    "review_id", "user_id", "business_id",
    "stars", "useful", "funny", "cool",
    "text", "review_timestamp",
    "is_latest_review"
]

# Truncate first, then append — preserves DDL schema & constraints
spark.sql("TRUNCATE TABLE yelp_poc_uc.silver_cleaned.yelp_review")

(
    df_flagged
    .select(*SILVER_COLUMNS)
    .write
    .mode("append")
    .saveAsTable("yelp_poc_uc.silver_cleaned.yelp_review")
)
print("✅ Silver Cleaned table populated: yelp_poc_uc.silver_cleaned.yelp_review")

In [0]:
# ============================================================
# ✅ Phase 6: Post-Write Validation
# ============================================================
# Read back from the persisted silver_cleaned table to verify.

silver_table = spark.read.table("yelp_poc_uc.silver_cleaned.yelp_review")
sv_count = silver_table.count()

print("=" * 65)
print("  ✅ Post-Write Validation Report")
print("=" * 65)

# 1. Row count reconciliation (Silver + ALL Quarantine tables = Bronze)
q_null_text  = spark.read.table("yelp_poc_uc.quarantine.yelp_review_null_text").count()
q_collision  = spark.read.table("yelp_poc_uc.quarantine.yelp_review_id_collision").count()
q_orphan     = spark.read.table("yelp_poc_uc.quarantine.yelp_review_orphan_user").count()
q_orphan_biz = spark.read.table("yelp_poc_uc.quarantine.yelp_review_orphan_business").count()   # Phase 4.5 (DQ-028)
q_total      = q_null_text + q_collision + q_orphan + q_orphan_biz

print(f"\n  1️⃣  Row Count Reconciliation")
print(f"     Bronze source:               {bronze_count:>12,}")
print(f"     Silver written:              {sv_count:>12,}")
print(f"     Quarantined (NULL text):     {q_null_text:>12,}")
print(f"     Quarantined (ID collision):  {q_collision:>12,}")
print(f"     Quarantined (orphan user):   {q_orphan:>12,}")
print(f"     Quarantined (orphan business):{q_orphan_biz:>11,}")
print(f"     Total quarantined:           {q_total:>12,}")
print(f"     Silver + Quarantine:         {sv_count + q_total:>12,}")
assert sv_count + q_total == bronze_count, f"❌ Row count mismatch! Bronze={bronze_count}, Silver+Quarantine={sv_count + q_total}"
print("     ✅ Silver + ALL Quarantine = Bronze — fully reconciled\n")

# 1b. Business FK integrity (DQ-REVIEW-028): every review has a parent in silver yelp_business
_sv_biz_ids = spark.read.table("yelp_poc_uc.silver_cleaned.yelp_business").select("business_id")
orphan_biz_in_silver = silver_table.join(_sv_biz_ids, "business_id", "left_anti").count()
print(f"  1️⃣b Business FK: reviews without a parent business = {orphan_biz_in_silver}")
assert orphan_biz_in_silver == 0, "❌ Silver reviews point to businesses not in silver_cleaned.yelp_business!"
print("     ✅ Every review has a parent business\n")

# 2. Primary key uniqueness (review_id)
distinct_pk = silver_table.select("review_id").distinct().count()
assert distinct_pk == sv_count, "❌ review_id is not unique!"
print(f"  2️⃣  PK Uniqueness: {distinct_pk:,} distinct review_ids == {sv_count:,} rows")
print("     ✅ review_id is unique\n")

# 3. Null checks on required columns
print("  3️⃣  Null Checks (required columns):")
for c in ["review_id", "user_id", "business_id", "stars", "review_timestamp", "text"]:
    null_ct = silver_table.filter(col(c).isNull()).count()
    status = "✅" if null_ct == 0 else "❌"
    print(f"     {status} {c}: {null_ct} nulls")
    assert null_ct == 0, f"❌ Nulls found in {c}!"
print()

# 4. Vote column range check (≥ 0)
print("  4️⃣  Vote Range Check (all ≥ 0):")
for c in ["useful", "funny", "cool"]:
    min_val = int(silver_table.agg(_min(c)).collect()[0][0])
    max_val = int(silver_table.agg(_max(c)).collect()[0][0])
    status = "✅" if min_val >= 0 else "❌"
    print(f"     {status} {c}: min={min_val}, max={max_val}")
    assert min_val >= 0, f"❌ {c} has values < 0!"
print()

# 5. Stars range check (1–5)
stars_min = int(silver_table.agg(_min("stars")).collect()[0][0])
stars_max = int(silver_table.agg(_max("stars")).collect()[0][0])
print(f"  5️⃣  Stars Range: min={stars_min}, max={stars_max}")
assert 1 <= stars_min and stars_max <= 5, "❌ stars outside 1–5!"
print("     ✅ Stars in valid range [1, 5]\n")

# 6. Flag distributions (only flags remaining in Silver)
print("  6️⃣  Flag Distributions:")
for flag in ["is_latest_review"]:
    true_ct  = silver_table.filter(col(flag) == True).count()
    false_ct = silver_table.filter(col(flag) == False).count()
    print(f"     {flag}: True={true_ct:,}  False={false_ct:,}")
print()

# 7. Schema verification
print("  7️⃣  Silver Cleaned Table Schema:")
silver_table.printSchema()

# 8. Column Constraints (PK, FK, NOT NULL)
print("\n  8️⃣  Column Constraints (PK / FK / NOT NULL):")
constraints_df = spark.sql("""
    SELECT
        column_name,
        data_type,
        is_nullable,
        comment
    FROM system.information_schema.columns
    WHERE table_catalog = 'yelp_poc_uc'
      AND table_schema  = 'silver_cleaned'
      AND table_name    = 'yelp_review'
    ORDER BY ordinal_position
""")
print("\n  ── Column Details (nullability) ──")
for row in constraints_df.collect():
    nullable_str = "NULLABLE" if row["is_nullable"] == "YES" else "NOT NULL"
    print(f"     {row['column_name']:25s} {row['data_type']:15s} {nullable_str}")

# Table constraints (PRIMARY KEY, FOREIGN KEY, etc.)
print("\n  ── Table Constraints (PK / FK / UNIQUE / CHECK) ──")
tc_df = spark.sql("""
    SELECT
        tc.constraint_name,
        tc.constraint_type,
        kcu.column_name
    FROM system.information_schema.table_constraints tc
    LEFT JOIN system.information_schema.constraint_column_usage kcu
        ON  tc.constraint_catalog = kcu.constraint_catalog
        AND tc.constraint_schema  = kcu.constraint_schema
        AND tc.constraint_name    = kcu.constraint_name
    WHERE tc.table_catalog = 'yelp_poc_uc'
      AND tc.table_schema  = 'silver_cleaned'
      AND tc.table_name    = 'yelp_review'
    ORDER BY tc.constraint_type, tc.constraint_name
""")
tc_rows = tc_df.collect()
if tc_rows:
    for row in tc_rows:
        col_name = row["column_name"] or "(table-level)"
        print(f"     {row['constraint_type']:15s} {row['constraint_name']:40s} → {col_name}")
else:
    print("     (no PK/FK/UNIQUE/CHECK constraints found on this table)")

# Also show CHECK constraints via DESCRIBE EXTENDED
print("\n  ── DESCRIBE DETAIL ──")
detail_df = spark.sql("DESCRIBE DETAIL yelp_poc_uc.silver_cleaned.yelp_review")
detail_df.select("format", "location", "numFiles", "sizeInBytes").show(truncate=False)

print("\n" + "=" * 65)
print("  🎉 All validations passed — Silver Cleaned review table is ready!")
print("=" * 65)